# Synthetic Banking Transaction Dataset for Fraud Detection

## Machine Learning Pipeline for Detecting Fraudulent Transactions

---

## Project Overview

This project focuses on building a machine learning pipeline for detecting fraudulent banking transactions using a synthetic transaction dataset.

The dataset contains transaction-level information related to customers, cards, merchants, devices, transaction amounts, locations, and transaction timing. It also includes behavioral and historical features that can help identify unusual transaction patterns.

The main objective of this project is to develop a classification model capable of distinguishing between legitimate and potentially fraudulent transactions.

---

## Project Objectives

### 1. Data Exploration and Understanding

* Explore the structure and characteristics of the dataset.
* Analyze numerical and categorical features.
* Investigate the distribution of legitimate and fraudulent transactions.
* Identify potential patterns and relationships within the data.

### 2. Feature Engineering

* Create time-based features from transaction timestamps.
* Extract customer transaction behavior.
* Calculate transaction frequency and historical spending features.
* Generate features that capture unusual transaction behavior.

### 3. Data Preprocessing

* Separate numerical and categorical features.
* Handle categorical variables through appropriate encoding techniques.
* Prepare the data for machine learning algorithms.
* Split the dataset into training, validation, and test sets.

### 4. Model Development

* Train machine learning classification models.
* Use the engineered features to detect potentially fraudulent transactions.
* Compare model performance using appropriate evaluation metrics.

### 5. Model Evaluation

The models will be evaluated using classification metrics such as:

* Precision
* Recall
* F1-Score
* ROC-AUC
* Confusion Matrix

Special attention will be given to fraud detection performance, since fraudulent transactions typically represent a small portion of the overall dataset.

## 1. Import Libraries

In [1]:
from urllib.parse import quote_plus
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

## 2. Connect to the Database

In [2]:
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_HOST = os.environ["DB_HOST"]
DB_PORT = "1433"
DB_NAME = "synthetic_banking_transaction_datawarehouse"

DB_DRIVER = "ODBC Driver 17 for SQL Server"

connection_params = (
    f"DRIVER={{{DB_DRIVER}}};"
    f"SERVER={DB_HOST},{DB_PORT};"
    f"DATABASE={DB_NAME};"
    f"UID={DB_USER};"
    f"PWD={DB_PASSWORD};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
)

DATABASE_URL = (
    f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_params)}"
)

engine = create_engine(
    DATABASE_URL,
    fast_executemany=True
)

## 3. Overview on Dataset

In [3]:
query = """
SELECT *
FROM dw.vw_fraud_ml
"""

df = pd.read_sql(query, engine)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 25 columns):
 #   Column              Non-Null Count    Dtype              
---  ------              --------------    -----              
 0   transaction_id      1000000 non-null  str                
 1   timestamp           1000000 non-null  str                
 2   customer_id         1000000 non-null  str                
 3   card_id             1000000 non-null  str                
 4   device_id           1000000 non-null  str                
 5   ip_address          1000000 non-null  str                
 6   merchant_id         1000000 non-null  str                
 7   merchant_category   1000000 non-null  str                
 8   merchant_country    1000000 non-null  str                
 9   merchant_city       1000000 non-null  str                
 10  merchant_latitude   1000000 non-null  float64            
 11  merchant_longitude  1000000 non-null  float64            
 12  transaction_

### 3.1. Count of is_fraud

In [4]:
print(df["is_fraud"].value_counts())
print()
print(df["is_fraud"].value_counts(normalize=True) * 100)

is_fraud
0    998510
1      1490
Name: count, dtype: int64

is_fraud
0    99.851
1     0.149
Name: proportion, dtype: float64


### 3.2. Duplicate

In [5]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate transaction_id:", df["transaction_id"].duplicated().sum())

Duplicate rows: 0
Duplicate transaction_id: 0


### 3.3. Missing values

In [6]:
missing = df.isna().sum()
print(missing[missing > 0])

fraud_type    998510
dtype: int64


### 3.4. Cardinality (Unique Values) for One-Hoting

In [7]:
categorical_cols = [
    "customer_id",
    "card_id",
    "device_id",
    "ip_address",
    "merchant_id",
    "merchant_category",
    "merchant_country",
    "merchant_city",
    "transaction_type",
    "currency",
    "day_of_week"
]

for col in categorical_cols:
    print(f"{col}: {df[col].nunique():,} unique values")

customer_id: 99,996 unique values
card_id: 238,624 unique values
device_id: 196,942 unique values
ip_address: 980,372 unique values
merchant_id: 20,080 unique values
merchant_category: 11 unique values
merchant_country: 10 unique values
merchant_city: 13,333 unique values
transaction_type: 2 unique values
currency: 1 unique values
day_of_week: 7 unique values


### 3.5. Numerical Features Descriptive Statistics

In [8]:
numeric_cols = [
    "amount",
    "merchant_latitude",
    "merchant_longitude",
    "transaction_hour",
    "transaction_minute",
    "day_of_month",
    "transaction_month",
    "transaction_year",
    "is_night"
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
amount,1000000.0,38.200372,77.689601,1.000000,9.860000,20.090000,40.930000,5000.000000
merchant_latitude,1000000.0,5.010033,37.486230,-59.995071,-27.350758,4.806735,37.685851,69.992733
merchant_longitude,1000000.0,1.220581,102.756189,-179.992484,-86.056396,1.297304,89.690926,179.973937
transaction_hour,1000000.0,11.499770,6.918451,0.000000,6.000000,11.000000,17.000000,23.000000
transaction_minute,1000000.0,29.493858,17.329639,0.000000,14.000000,29.000000,45.000000,59.000000
day_of_month,1000000.0,15.728885,8.795952,1.000000,8.000000,16.000000,23.000000,31.000000
transaction_month,1000000.0,6.527620,3.448710,1.000000,4.000000,7.000000,10.000000,12.000000
transaction_year,1000000.0,2024.829847,0.375767,2024.000000,2025.000000,2025.000000,2025.000000,2025.000000
is_night,1000000.0,0.249979,0.433001,0.000000,0.000000,0.000000,0.000000,1.000000


### 3.6. Transaction Amount Distribution by Fraud Status

In [9]:
df.groupby("is_fraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,998510.0,38.220177,77.693725,1.0,9.88,20.10,40.9500,5000.00
1,1490.0,24.928671,73.712312,1.0,1.64,2.57,22.8025,1794.48


### 3.7. Fraud Rate by Transaction Hour

In [10]:
hourly_fraud = (
    df.groupby("transaction_hour")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

hourly_fraud["fraud_rate_pct"] = hourly_fraud["fraud_rate"] * 100

hourly_fraud

,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
transaction_hour,,,,
0,41059,55,0.001340,0.133954
1,42109,66,0.001567,0.156736
2,41632,76,0.001826,0.182552
3,41390,55,0.001329,0.132882
4,42237,60,0.001421,0.142056
5,41552,73,0.001757,0.175683
6,41768,48,0.001149,0.114921
7,41115,48,0.001167,0.116746
8,41722,70,0.001678,0.167777


### 3.8. Fraud Rate by Day of Week

In [11]:
daily_fraud = (
    df.groupby("day_of_week")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

daily_fraud["fraud_rate_pct"] = daily_fraud["fraud_rate"] * 100

daily_fraud

,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
day_of_week,,,,
Friday,142469,213,0.001495,0.149506
Monday,142287,212,0.001490,0.148995
Saturday,142227,231,0.001624,0.162416
Sunday,143270,229,0.001598,0.159838
Thursday,144265,199,0.001379,0.137941
Tuesday,142918,191,0.001336,0.133643
Wednesday,142564,215,0.001508,0.150809


### 3.9. Fraud Rate by Nighttime Status

In [12]:
night_fraud = (
    df.groupby("is_night")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

night_fraud["fraud_rate_pct"] = night_fraud["fraud_rate"] * 100

night_fraud

,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
is_night,,,,
0,750021,1105,0.001473,0.147329
1,249979,385,0.001540,0.154013


### 3.10. Fraud Rate by Merchant Category

In [13]:
merchant_category_fraud = (
    df.groupby("merchant_category")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
      .sort_values("fraud_rate", ascending=False)
)

merchant_category_fraud["fraud_rate_pct"] = (
    merchant_category_fraud["fraud_rate"] * 100
)

merchant_category_fraud

,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
merchant_category,,,,
transfer,80,80,1.000000,100.000000
online_marketplace,29690,62,0.002088,0.208825
entertainment,106326,164,0.001542,0.154243
fashion,108131,165,0.001526,0.152593
electronics,108315,160,0.001477,0.147717
pharmacy,107811,158,0.001466,0.146553
utilities,109210,155,0.001419,0.141928
restaurants,108612,149,0.001372,0.137186
grocery,108950,140,0.001285,0.128499


### 3.11. Merchant Category and Transaction Type Relationship

In [14]:
pd.crosstab(
    df["merchant_category"],
    df["transaction_type"]
)

transaction_type,purchase,transfer
merchant_category,,
electronics,108315,0
entertainment,106326,0
fashion,108131,0
gas_station,106974,0
grocery,108950,0
online_marketplace,29690,0
pharmacy,107811,0
restaurants,108612,0
transfer,0,80


### 3.12. Fraud Rate by Merchant Country

In [15]:
country_fraud = (
    df.groupby("merchant_country")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
      .sort_values("fraud_rate", ascending=False)
)

country_fraud["fraud_rate_pct"] = country_fraud["fraud_rate"] * 100

country_fraud

,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
merchant_country,,,,
BR,103600,166,0.001602,0.160232
ES,99579,157,0.001577,0.157664
CA,100283,157,0.001566,0.156557
DE,97909,152,0.001552,0.155246
GB,98823,153,0.001548,0.154822
US,99087,146,0.001473,0.147345
IN,100417,143,0.001424,0.142406
IT,98973,140,0.001415,0.141453
FR,96174,133,0.001383,0.138291


### 3.13. Fraud Rate by Transaction Type

In [16]:
transaction_type_fraud = (
    df.groupby("transaction_type")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

transaction_type_fraud["fraud_rate_pct"] = (
    transaction_type_fraud["fraud_rate"] * 100
)

transaction_type_fraud

,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
transaction_type,,,,
purchase,999920,1410,0.00141,0.141011
transfer,80,80,1.00000,100.000000


## 4. Feature Selection and Dataset Preparation

### 4.1. Create df_model and Column Removal

In [17]:
drop_cols = [
    "transaction_id",
    "fraud_type",
    "currency"
]

df_model = df.drop(columns=drop_cols)

### 4.2. Sort timestamp for training

In [18]:
df_model = df_model.sort_values("timestamp").reset_index(drop=True)

## 5. Feature Engineering

### 5.1. Fraud Rate by Weekend Status

In [19]:
df_model["is_weekend"] = (
    df_model["day_of_week"].isin(["Saturday", "Sunday"])
).astype(int)

df_model["is_weekend"].value_counts()

is_weekend
0    714503
1    285497
Name: count, dtype: int64

In [20]:
weekend_fraud = (
    df_model.groupby("is_weekend")["is_fraud"]
    .agg(
        transaction_count="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

weekend_fraud["fraud_rate_pct"] = weekend_fraud["fraud_rate"] * 100

weekend_fraud

,transaction_count,fraud_count,fraud_rate,fraud_rate_pct
is_weekend,,,,
0,714503,1030,0.001442,0.144156
1,285497,460,0.001611,0.161123


### 5.2. Customer Transaction Frequency in the Previous 24 Hours

In [21]:
# Make sure timestamp is in datetime format
df_model["timestamp"] = pd.to_datetime(
    df_model["timestamp"],
    utc=True
)

# Sort by customer and timestamp
df_model = df_model.sort_values(
    ["customer_id", "timestamp"]
).reset_index(drop=True)

# Convert timestamp to int64 nanoseconds
time_ns = df_model["timestamp"].astype("int64").to_numpy()

# 24 hours in nanoseconds
window_24h = 24 * 60 * 60 * 1_000_000_000

# Result array
customer_txn_count_24h = np.zeros(
    len(df_model),
    dtype=np.int32
)

# Process each customer
for _, idx in df_model.groupby("customer_id").groups.items():

    positions = np.asarray(idx)
    customer_times = time_ns[positions]

    # Find the first transaction within the previous 24 hours
    left_positions = np.searchsorted(
        customer_times,
        customer_times - window_24h,
        side="left"
    )

    # Count previous transactions
    customer_txn_count_24h[positions] = (
        np.arange(len(positions)) - left_positions
    )

# Add feature
df_model["customer_txn_count_24h"] = customer_txn_count_24h

In [22]:
df_model["customer_txn_count_24h"].describe()

count    1000000.000000
mean           4.998687
std            3.649632
min            0.000000
25%            2.000000
50%            5.000000
75%            7.000000
max           26.000000
Name: customer_txn_count_24h, dtype: float64

In [23]:
df_model.groupby("is_fraud")["customer_txn_count_24h"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,998510.0,4.998474,3.649334,0.0,2.0,5.0,7.0,26.0
1,1490.0,5.141611,3.842528,0.0,2.0,5.0,8.0,21.0


### 5.3. Time Since Last Customer Transaction

In [24]:
# Sort by customer and timestamp
df_model = df_model.sort_values(
    ["customer_id", "timestamp"]
).reset_index(drop=True)

# Calculate time since the previous transaction
df_model["time_since_last_customer_txn"] = (
    df_model.groupby("customer_id")["timestamp"]
    .diff()
    .dt.total_seconds()
    / 60
)

# First transaction of each customer has no previous transaction
df_model["time_since_last_customer_txn"] = (
    df_model["time_since_last_customer_txn"]
    .fillna(-1)
)

In [25]:
df_model["time_since_last_customer_txn"].describe()

count    1000000.000000
mean       42075.615054
std        45927.470409
min           -1.000000
25%         8622.662500
50%        27687.741667
75%        60215.570833
max       510693.550000
Name: time_since_last_customer_txn, dtype: float64

In [26]:
df_model.groupby("is_fraud")["time_since_last_customer_txn"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,998510.0,42076.993588,45929.376596,-1.0,8623.816667,27687.041667,60217.341667,510693.55
1,1490.0,41151.803579,44637.159233,-1.0,7904.920833,28230.100000,59174.791667,311019.55


### 5.4. Customer Average Transaction Amount in the Previous 7 Days

In [27]:
df_model = df_model.sort_values(
    ["customer_id", "timestamp"]
).reset_index(drop=True)


def customer_avg_amount_7d(group):
    amount_series = (
        group
        .set_index("timestamp")["amount"]
        .rolling("7D", closed="left")
        .mean()
    )

    amount_series.index = group.index
    return amount_series


df_model["customer_avg_amount_7d"] = (
    df_model
    .groupby("customer_id", group_keys=False)
    .apply(customer_avg_amount_7d)
    .reset_index(level=0, drop=True)
)

In [28]:
df_model["customer_avg_amount_7d"].describe()

count    173113.000000
mean         37.997467
std          72.924208
min           1.000000
25%          10.100000
50%          20.370000
75%          41.140000
max        5000.000000
Name: customer_avg_amount_7d, dtype: float64

In [29]:
df_model.groupby("is_fraud")["customer_avg_amount_7d"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,172851.0,37.977783,72.532564,1.0,10.1,20.37,41.14000,5000.00
1,262.0,50.983896,207.113253,1.0,10.5,20.76,40.69375,3224.67


### 5.5. Transaction Amount Relative to Customer's 7-Day Average

In [30]:
df_model["amount_vs_customer_avg_7d"] = (
    df_model["amount"] / df_model["customer_avg_amount_7d"]
)

In [31]:
df_model["amount_vs_customer_avg_7d"].describe()

count    173113.000000
mean          2.244142
std           7.739895
min           0.000567
25%           0.496614
50%           0.985948
75%           1.955657
max        1084.381818
Name: amount_vs_customer_avg_7d, dtype: float64

In [32]:
df_model.groupby("is_fraud")["amount_vs_customer_avg_7d"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,172851.0,2.245372,7.744037,0.000583,0.497447,0.986635,1.956686,1084.381818
1,262.0,1.432317,4.122784,0.000567,0.085334,0.308601,1.220519,41.097473


### 5.6. Card Transaction Frequency in the Previous 24 Hours

In [33]:
df_model = df_model.sort_values(
    ["card_id", "timestamp"]
).reset_index(drop=True)

# Convert timestamp to int64 nanoseconds
time_ns = df_model["timestamp"].astype("int64").to_numpy()

# 24-hour window in nanoseconds
window_24h = 24 * 60 * 60 * 1_000_000_000

# Result array
card_txn_count_24h = np.zeros(
    len(df_model),
    dtype=np.int32
)

# Process each card
for _, idx in df_model.groupby("card_id").groups.items():

    positions = np.asarray(idx)
    card_times = time_ns[positions]

    # Find the first transaction within the previous 24 hours
    left_positions = np.searchsorted(
        card_times,
        card_times - window_24h,
        side="left"
    )

    # Count previous transactions
    card_txn_count_24h[positions] = (
        np.arange(len(positions)) - left_positions
    )

# Add feature
df_model["card_txn_count_24h"] = card_txn_count_24h

In [34]:
df_model["card_txn_count_24h"].describe()

count    1000000.000000
mean           2.611376
std            2.785982
min            0.000000
25%            1.000000
50%            2.000000
75%            4.000000
max           26.000000
Name: card_txn_count_24h, dtype: float64

In [35]:
df_model.groupby("is_fraud")["card_txn_count_24h"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,998510.0,2.608431,2.782439,0.0,1.0,2.0,4.0,26.0
1,1490.0,4.585235,4.134222,0.0,1.0,4.0,7.0,22.0


### 5.7. Device Transaction Frequency in the Previous 24 Hours

In [36]:
df_model = df_model.sort_values(
    ["device_id", "timestamp"]
).reset_index(drop=True)

# Convert timestamp to int64 nanoseconds
time_ns = df_model["timestamp"].astype("int64").to_numpy()

# 24-hour window in nanoseconds
window_24h = 24 * 60 * 60 * 1_000_000_000

# Result array
device_txn_count_24h = np.zeros(
    len(df_model),
    dtype=np.int32
)

# Process each device
for _, idx in df_model.groupby("device_id").groups.items():

    positions = np.asarray(idx)
    device_times = time_ns[positions]

    # Find the first transaction within the previous 24 hours
    left_positions = np.searchsorted(
        device_times,
        device_times - window_24h,
        side="left"
    )

    # Count previous transactions
    device_txn_count_24h[positions] = (
        np.arange(len(positions)) - left_positions
    )

# Add feature
df_model["device_txn_count_24h"] = device_txn_count_24h

In [37]:
df_model["device_txn_count_24h"].describe()

count    1000000.000000
mean           3.046956
std            2.966309
min            0.000000
25%            1.000000
50%            2.000000
75%            4.000000
max           26.000000
Name: device_txn_count_24h, dtype: float64

In [38]:
df_model.groupby("is_fraud")["device_txn_count_24h"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,998510.0,3.050035,2.966271,0.0,1.0,2.0,4.0,26.0
1,1490.0,0.983893,2.165982,0.0,0.0,0.0,1.0,16.0


### 5.8. IP Transaction Frequency in the Previous 24 Hours

In [39]:
# Sort by IP address and timestamp
df_model = df_model.sort_values(
    ["ip_address", "timestamp"]
).reset_index(drop=True)

# Convert timestamp to int64 nanoseconds
time_ns = df_model["timestamp"].astype("int64").to_numpy()

# 24-hour window in nanoseconds
window_24h = 24 * 60 * 60 * 1_000_000_000

# Result array
ip_txn_count_24h = np.zeros(
    len(df_model),
    dtype=np.int32
)

# Process each IP address
for _, idx in df_model.groupby("ip_address").groups.items():

    positions = np.asarray(idx)
    ip_times = time_ns[positions]

    # Find the first transaction within the previous 24 hours
    left_positions = np.searchsorted(
        ip_times,
        ip_times - window_24h,
        side="left"
    )

    # Count previous transactions
    ip_txn_count_24h[positions] = (
        np.arange(len(positions)) - left_positions
    )

# Add feature
df_model["ip_txn_count_24h"] = ip_txn_count_24h

In [40]:
df_model["ip_txn_count_24h"].describe()

count    1000000.000000
mean           0.019866
std            0.141256
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            3.000000
Name: ip_txn_count_24h, dtype: float64

In [41]:
df_model.groupby("is_fraud")["ip_txn_count_24h"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,998510.0,0.019860,0.141237,0.0,0.0,0.0,0.0,3.0
1,1490.0,0.024161,0.153601,0.0,0.0,0.0,0.0,1.0


## 6. Feature Selection and Cleanup

In [42]:
# Remove weak feature
df_model = df_model.drop(
    columns=["ip_txn_count_24h"]
)

## 7. Time-Based Train, Validation, and Test Split

### 7.1. Split to Train, Validation and Test

In [43]:
df_model = df_model.sort_values(
    "timestamp"
).reset_index(drop=True)

# Calculate split points
n = len(df_model)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Create time-based splits
train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

### 7.2. Shape of Datasets

In [44]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (700000, 29)
Validation: (150000, 29)
Test: (150000, 29)


### 7.3. Fraud Distribution

In [45]:
split_summary = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "Rows": [
        len(train_df),
        len(val_df),
        len(test_df)
    ],
    "Fraud": [
        train_df["is_fraud"].sum(),
        val_df["is_fraud"].sum(),
        test_df["is_fraud"].sum()
    ]
})

split_summary["Fraud Rate (%)"] = (
    split_summary["Fraud"] /
    split_summary["Rows"] * 100
)

split_summary

,Dataset,Rows,Fraud,Fraud Rate (%)
0,Train,700000,1056,0.150857
1,Validation,150000,219,0.146000
2,Test,150000,215,0.143333


### 7.4. Verify Time-Based Data Split

In [46]:
print("Train:")
print(train_df["timestamp"].min(), "->", train_df["timestamp"].max())

print("\nValidation:")
print(val_df["timestamp"].min(), "->", val_df["timestamp"].max())

print("\nTest:")
print(test_df["timestamp"].min(), "->", test_df["timestamp"].max())

Train:
2024-10-30 20:56:15+00:00 -> 2025-07-13 13:43:41+00:00

Validation:
2025-07-13 13:44:05+00:00 -> 2025-09-06 06:20:28+00:00

Test:
2025-09-06 06:20:41+00:00 -> 2025-10-30 20:54:55+00:00


## 8. Feature and Data Preparation

### 8.1. Remove Non-Model Features

In [47]:
drop_model_cols = [
    "timestamp",
    "transaction_time",
    "is_fraud",
    "customer_id",
    "card_id",
    "device_id",
    "ip_address",
    "merchant_id"
]

X_train = train_df.drop(columns=drop_model_cols)
y_train = train_df["is_fraud"]

X_val = val_df.drop(columns=drop_model_cols)
y_val = val_df["is_fraud"]

X_test = test_df.drop(columns=drop_model_cols)
y_test = test_df["is_fraud"]

In [48]:
print("Features:")
print(X_train.columns.tolist())

print("\nNumber of features:", X_train.shape[1])

Features:
['merchant_category', 'merchant_country', 'merchant_city', 'merchant_latitude', 'merchant_longitude', 'transaction_type', 'amount', 'transaction_hour', 'transaction_minute', 'day_of_week', 'day_of_month', 'transaction_month', 'transaction_year', 'is_night', 'is_weekend', 'customer_txn_count_24h', 'time_since_last_customer_txn', 'customer_avg_amount_7d', 'amount_vs_customer_avg_7d', 'card_txn_count_24h', 'device_txn_count_24h']

Number of features: 21


In [49]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (700000, 21)
X_val: (150000, 21)
X_test: (150000, 21)

y_train: (700000,)
y_val: (150000,)
y_test: (150000,)


### 8.2. Categorical Feature Encoding

In [50]:
categorical_cols = [
    "merchant_category",
    "merchant_country",
    "transaction_type",
    "day_of_week"
]

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(
    X_train[categorical_cols]
)

X_val_cat = encoder.transform(
    X_val[categorical_cols]
)

X_test_cat = encoder.transform(
    X_test[categorical_cols]
)

In [51]:
print("Encoded Train shape:", X_train_cat.shape)
print("Encoded Validation shape:", X_val_cat.shape)
print("Encoded Test shape:", X_test_cat.shape)

Encoded Train shape: (700000, 30)
Encoded Validation shape: (150000, 30)
Encoded Test shape: (150000, 30)


### 8.3. Numerical Feature Encoding

In [52]:
numerical_cols = [
    "merchant_latitude",
    "merchant_longitude",
    "amount",
    "transaction_hour",
    "transaction_minute",
    "day_of_month",
    "transaction_month",
    "transaction_year",
    "is_night",
    "is_weekend",
    "customer_txn_count_24h",
    "time_since_last_customer_txn",
    "customer_avg_amount_7d",
    "amount_vs_customer_avg_7d",
    "card_txn_count_24h",
    "device_txn_count_24h"
]

X_train_num = X_train[numerical_cols].to_numpy()
X_val_num = X_val[numerical_cols].to_numpy()
X_test_num = X_test[numerical_cols].to_numpy()

### 8.4. Combining Categorical and Numerical Features

In [53]:
X_train_final = np.hstack([
    X_train_num,
    X_train_cat
])

X_val_final = np.hstack([
    X_val_num,
    X_val_cat
])

X_test_final = np.hstack([
    X_test_num,
    X_test_cat
])

In [54]:
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)

X_train_final: (700000, 46)
X_val_final: (150000, 46)
X_test_final: (150000, 46)


## 9. Training Phase

### 9.1. Handeling NaN values in Train set with Median

In [55]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(
    strategy="median"
)

X_train_imputed = imputer.fit_transform(X_train_final)

print("X_train_imputed:", X_train_imputed.shape)

X_train_imputed: (700000, 46)


### 9.2. Prepare Train Data

In [56]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)

print("X_train_scaled:", X_train_scaled.shape)

X_train_scaled: (700000, 46)


### 9.3. Prepare Validation Data

In [57]:
X_val_imputed = imputer.transform(X_val_final)

X_val_scaled = scaler.transform(X_val_imputed)

print("X_val_scaled:", X_val_scaled.shape)

X_val_scaled: (150000, 46)


### 9.4. Prepare Test Data

In [58]:
X_test_imputed = imputer.transform(X_test_final)

X_test_scaled = scaler.transform(X_test_imputed)

print("X_test_scaled:", X_test_scaled.shape)

X_test_scaled: (150000, 46)


### 9.5. Train Model with LightGBM

In [64]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(
    X_train_imputed,
    y_train
)

# LightGBM Validation Predictions
y_val_proba_lgbm = lgbm_model.predict_proba(
    X_val_imputed
)[:, 1]

print("LightGBM training completed.")

[LightGBM] [Info] Number of positive: 1056, number of negative: 698944
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021382 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1783
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM training completed.


### 9.6. LightGBM Threshold Analysis

In [66]:
thresholds = np.arange(0.01, 1.00, 0.01)

threshold_results_lgbm = []

for threshold in thresholds:
    y_val_pred = (
        y_val_proba_lgbm >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    threshold_results_lgbm.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results_lgbm = pd.DataFrame(
    threshold_results_lgbm
)

threshold_results_lgbm

,threshold,precision,recall,f1
0,0.01,0.001989,0.954338,0.003970
1,0.02,0.002648,0.913242,0.005280
2,0.03,0.003350,0.858447,0.006674
3,0.04,0.004229,0.821918,0.008415
4,0.05,0.005332,0.799087,0.010593
...,...,...,...,...
94,0.95,0.649533,0.634703,0.642032
95,0.96,0.668293,0.625571,0.646226
96,0.97,0.712766,0.611872,0.658477
97,0.98,0.735955,0.598174,0.659950


### 9.7. LightGBM Hyperparameter Tuning

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score
import optuna

# Previous Hyperparameters for the Previous Tuned Model: {'num_leaves': 53, 'max_depth': 10, 'learning_rate': 0.06336926793856841, 'n_estimators': 734, 'min_child_samples': 19}
# Best Hyperparameters for the Previous Tuned Model: {'num_leaves': 236, 'max_depth': 26, 'learning_rate': 0.014185095317721818, 'n_estimators': 909, 'min_child_samples': 30} , Best PR-AUC: 0.6652109460954154

def objective(trial):

    params = {

        "objective": "binary",
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,

        # ######################################################### #
        "num_leaves": trial.suggest_int(
            "num_leaves", 15, 300
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 5, 40
        ),
        # ######################################################### #

        # ######################################################### #
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.10, log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators", 700, 1000
        ),
        # ######################################################### #

        # ######################################################### #
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 10, 50
        )
        # ######################################################### #

    }

    model = LGBMClassifier(**params)

    model.fit(
        X_train_imputed,
        y_train
    )

    y_val_proba = model.predict_proba(
        X_val_imputed
    )[:, 1]

    pr_auc = average_precision_score(
        y_val,
        y_val_proba
    )

    return pr_auc


study = optuna.create_study(
    direction="maximize",
    study_name="lightgbm_fraud_detection"
)

study.optimize(
    objective,
    n_trials=20
)

[I 2026-10-02 13:36:18,925] A new study created in memory with name: lightgbm_fraud_detection


[LightGBM] [Info] Number of positive: 1056, number of negative: 698944
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027501 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1783
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[I 2026-10-02 13:37:03,106] Trial 0 finished with value: 0.6563390213561034 and parameters: {'num_leaves': 207, 'max_depth': 23, 'learning_rate': 0.0397905336011871, 'n_estimators': 952, 'min_child_samples': 45}. Best is trial 0 with value: 0.6563390213561034.


[LightGBM] [Info] Number of positive: 1056, number of negative: 698944
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021127 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1783
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[I 2026-10-02 13:37:52,133] Trial 1 finished with value: 0.6552544060207355 and parameters: {'num_leaves': 255, 'max_depth': 29, 'learning_rate': 0.03783954842059244, 'n_estimators': 882, 'min_child_samples': 50}. Best is trial 0 with value: 0.6563390213561034.


[LightGBM] [Info] Number of positive: 1056, number of negative: 698944
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025950 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1783
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


... (31 outputs left out)

[LightGBM] [Info] Number of positive: 1056, number of negative: 698944
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023827 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1783
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[I 2026-10-02 13:48:22,360] Trial 18 finished with value: 0.6652109460954154 and parameters: {'num_leaves': 236, 'max_depth': 26, 'learning_rate': 0.014185095317721818, 'n_estimators': 909, 'min_child_samples': 30}. Best is trial 18 with value: 0.6652109460954154.


[LightGBM] [Info] Number of positive: 1056, number of negative: 698944
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026643 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1783
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[I 2026-10-02 13:49:16,874] Trial 19 finished with value: 0.6585836280121925 and parameters: {'num_leaves': 247, 'max_depth': 24, 'learning_rate': 0.011001844461391614, 'n_estimators': 957, 'min_child_samples': 41}. Best is trial 18 with value: 0.6652109460954154.


### 9.8. Best Hyperparameters for tuning the Model

In [107]:
print("Best Trial:")
print(study.best_trial)

print("\nBest Parameters:")
print(study.best_params)

print("\nBest PR-AUC:")
print(study.best_value)

Best Trial:
FrozenTrial(number=18, state=<TrialState.COMPLETE: 1>, values=[0.6652109460954154], datetime_start=datetime.datetime(2026, 10, 2, 13, 47, 32, 569324), datetime_complete=datetime.datetime(2026, 10, 2, 13, 48, 22, 360660), params={'num_leaves': 236, 'max_depth': 26, 'learning_rate': 0.014185095317721818, 'n_estimators': 909, 'min_child_samples': 30}, user_attrs={}, system_attrs={'tpe:relative_params:0': '{"learning_rate": 0.014185095317721818, "max_depth": 26, "min_child_samples": 30, "n_estimators": 909, "num_leaves": 236}'}, intermediate_values={}, distributions={'num_leaves': IntDistribution(high=300, log=False, low=15, step=1), 'max_depth': IntDistribution(high=40, log=False, low=5, step=1), 'learning_rate': FloatDistribution(high=0.1, log=True, low=0.01, step=None), 'n_estimators': IntDistribution(high=1000, log=False, low=700, step=1), 'min_child_samples': IntDistribution(high=50, log=False, low=10, step=1)}, trial_id=18, value=None)

Best Parameters:
{'num_leaves': 236

### 9.9. Train Tuned LightGBM Model

In [108]:
tuned_lgbm_model = LGBMClassifier(

    objective="binary",

    num_leaves=236,

    max_depth=26,

    learning_rate=0.014185095317721818,

    n_estimators=909,

    min_child_samples=30,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1

)

tuned_lgbm_model.fit(

    X_train_imputed,

    y_train

)

print("Optuna-tuned LightGBM training completed.")

[LightGBM] [Info] Number of positive: 1056, number of negative: 698944
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025687 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1783
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Optuna-tuned LightGBM training completed.


### 9.10. Tuned LightGBM Validation PR-AUC

In [109]:
y_val_proba_tuned = tuned_lgbm_model.predict_proba(
    X_val_imputed
)[:, 1]

pr_auc_tuned = average_precision_score(
    y_val,
    y_val_proba_tuned
)

print(f"Tuned LightGBM PR-AUC: {pr_auc_tuned:.4f}")

Tuned LightGBM PR-AUC: 0.6652


### 9.11. Tuned LightGBM Threshold Analysis

In [110]:
thresholds = np.arange(0.01, 1.00, 0.01)

threshold_results_tuned = []

for threshold in thresholds:

    y_val_pred = (
        y_val_proba_tuned >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    threshold_results_tuned.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results_tuned = pd.DataFrame(
    threshold_results_tuned
)

threshold_results_tuned.loc[
    threshold_results_tuned["f1"].idxmax()
]

threshold    0.900000
precision    0.888158
recall       0.616438
f1           0.727763
Name: 89, dtype: float64

### 9.12. Tuned LightGBM Confusion Matrix

In [112]:
threshold = 0.90

y_val_pred_tuned = (
    y_val_proba_tuned >= threshold
).astype(int)

cm_tuned = confusion_matrix(
    y_val,
    y_val_pred_tuned
)

print("Confusion Matrix at Threshold = 0.90:")
print(cm_tuned)

Confusion Matrix at Threshold = 0.90:
[[149764     17]
 [    84    135]]


### 9.13. Tuned LightGBM Threshold Comparison

In [113]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

thresholds = np.arange(0.80, 1.00, 0.01)

threshold_results = []

for threshold in thresholds:
    y_val_pred = (
        y_val_proba_tuned >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_val_pred
    ).ravel()

    precision = precision_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    fpr = fp / (fp + tn)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "false_positives": fp,
        "false_negatives": fn
    })

threshold_results_df = pd.DataFrame(
    threshold_results
)

threshold_results_df

,threshold,precision,recall,f1,fpr,false_positives,false_negatives
0,0.80,0.814371,0.621005,0.704663,0.000207,31,83
1,0.81,0.814371,0.621005,0.704663,0.000207,31,83
2,0.82,0.819277,0.621005,0.706494,0.000200,30,83
3,0.83,0.829268,0.621005,0.710183,0.000187,28,83
4,0.84,0.834356,0.621005,0.712042,0.000180,27,83
5,0.85,0.844720,0.621005,0.715789,0.000167,25,83
6,0.86,0.844720,0.621005,0.715789,0.000167,25,83
7,0.87,0.860759,0.621005,0.721485,0.000147,22,83
8,0.88,0.865385,0.616438,0.720000,0.000140,21,84
9,0.89,0.876623,0.616438,0.723861,0.000127,19,84


### 9.14. Final Validation Results

In [115]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

threshold = 0.90

y_val_pred_final = (
    y_val_proba_tuned >= threshold
).astype(int)

precision = precision_score(
    y_val,
    y_val_pred_final
)

recall = recall_score(
    y_val,
    y_val_pred_final
)

f1 = f1_score(
    y_val,
    y_val_pred_final
)

tn, fp, fn, tp = confusion_matrix(
    y_val,
    y_val_pred_final
).ravel()

fpr = fp / (fp + tn)

print(f"Threshold: {threshold:.2f}")
print(f"PR-AUC: {pr_auc_tuned:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"False Positive Rate: {fpr:.6f}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Negatives: {tn}")

Threshold: 0.90
PR-AUC: 0.6652
Precision: 0.8882
Recall: 0.6164
F1-score: 0.7278
False Positive Rate: 0.000113
True Positives: 135
False Positives: 17
False Negatives: 84
True Negatives: 149764


## 10. Test Set Evaluation

### 10.1. Prepare Test Evaluation

In [116]:
y_test_proba = tuned_lgbm_model.predict_proba(
    X_test_imputed
)[:, 1]

print("Test probabilities generated.")

Test probabilities generated.


### 10.2. Apply Validation Threshold to Test

In [117]:
selected_threshold = 0.90

y_test_pred = (
    y_test_proba >= selected_threshold
).astype(int)

print("Test predictions generated.")

Test predictions generated.


### 10.3. Test Confusion Matrix 

In [119]:
from sklearn.metrics import confusion_matrix

cm_test = confusion_matrix(
    y_test,
    y_test_pred
)

print("Confusion Matrix at Threshold = 0.90:")
print(cm_test)

Confusion Matrix at Threshold = 0.90:
[[149763     22]
 [    94    121]]


### 10.4. Final Test Metrics

In [120]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

test_pr_auc = average_precision_score(
    y_test,
    y_test_proba
)

test_precision = precision_score(
    y_test,
    y_test_pred
)

test_recall = recall_score(
    y_test,
    y_test_pred
)

test_f1 = f1_score(
    y_test,
    y_test_pred
)

tn, fp, fn, tp = cm_test.ravel()

test_fpr = fp / (fp + tn)

print(f"Threshold: {selected_threshold}")
print(f"PR-AUC: {test_pr_auc:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1-score: {test_f1:.4f}")
print(f"False Positive Rate: {test_fpr:.6f}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Negatives: {tn}")

Threshold: 0.9
PR-AUC: 0.6238
Precision: 0.8462
Recall: 0.5628
F1-score: 0.6760
False Positive Rate: 0.000147
True Positives: 121
False Positives: 22
False Negatives: 94
True Negatives: 149763


# Final Model Results and Conclusion

## Final Model

The final fraud detection model is a **LightGBM classifier fine-tuned using Optuna**.

The model was trained using a time-based split:

* **70%** Training set
* **15%** Validation set
* **15%** Test set

The final model uses the following hyperparameters:

* `num_leaves = 236`
* `max_depth = 26`
* `learning_rate = 0.0141850953`
* `n_estimators = 909`
* `min_child_samples = 30`
* `class_weight = "balanced"`

The classification threshold was selected on the Validation set based on the F1-score and fixed at **0.90** before evaluating the Test set.

## Final Test Results

| Metric              |  Test Result |
| ------------------- | -----------: |
| PR-AUC              |   **0.6238** |
| Precision           |   **0.8462** |
| Recall              |   **0.5628** |
| F1-score            |   **0.6760** |
| False Positive Rate | **0.000147** |
| True Positives      |      **121** |
| False Positives     |       **22** |
| False Negatives     |       **94** |
| True Negatives      |  **149,763** |

## Interpretation

The final model achieved a **Precision of 84.62%**, meaning that most transactions classified as fraudulent were actually fraudulent.

The model achieved a **Recall of 56.28%**, detecting 121 out of 215 fraudulent transactions in the Test set. However, 94 fraudulent transactions were not detected.

The **False Positive Rate was only 0.0147%**, with just 22 legitimate transactions incorrectly classified as fraudulent. This is particularly relevant for a highly imbalanced fraud detection problem, where fraudulent transactions represent only a very small proportion of all transactions.

The final **PR-AUC of 0.6238** indicates that the LightGBM model achieved substantially better fraud-ranking performance than the Logistic Regression baseline, which achieved a PR-AUC of 0.2112.

## Validation vs Test

| Metric              | Validation |         Test |
| ------------------- | ---------: | -----------: |
| PR-AUC              |     0.6652 |   **0.6238** |
| Precision           |     0.8882 |   **0.8462** |
| Recall              |     0.6164 |   **0.5628** |
| F1-score            |     0.7278 |   **0.6760** |
| False Positive Rate |   0.000113 | **0.000147** |

The difference between Validation and Test performance indicates some reduction in performance on unseen data. However, the model maintained relatively high Precision and a very low False Positive Rate on the Test set.

## Comparison with the Previous Tuned Model

Compared with the previous LightGBM model, the final fine-tuned model achieved improved Test performance:

| Metric              | Previous Model |  Final Model |
| ------------------- | -------------: | -----------: |
| PR-AUC              |         0.5921 |   **0.6238** |
| Precision           |         0.8041 |   **0.8462** |
| Recall              |         0.5535 |   **0.5628** |
| F1-score            |         0.6556 |   **0.6760** |
| False Positive Rate |       0.000194 | **0.000147** |

The final model improved PR-AUC, Precision, Recall, and F1-score while also reducing the False Positive Rate.

## Conclusion

The final LightGBM model provides a practical approach for detecting fraudulent banking transactions in this synthetic dataset. It achieves relatively high Precision and a very low False Positive Rate while detecting more than half of the fraudulent transactions.

The main limitation is the **56.28% Recall**, meaning that a considerable number of fraudulent transactions are still missed. Therefore, future improvements could focus on increasing fraud detection coverage while maintaining an acceptable false positive rate.

Overall, the project demonstrates an end-to-end fraud detection pipeline covering data exploration, feature engineering, time-based data splitting, preprocessing, baseline modeling, LightGBM modeling, hyperparameter optimization, threshold selection, and final evaluation on an unseen Test set.
